# Holosoma Retargeting Pipeline

## 1. 使用的模型资源类型
- URDF （Unified Robot Description Format）： 主要用来描述机器人由哪些刚体 link 组成，这些 link 之间用什么 joint 连接。
- MuJoCo XML 是优化和碰撞计算实际加载的场景
- OBJ/STL/DAE 是被 URDF 或 XML 引用的网格资源。

# 2. 主流程和数据流

`robot_retarget.py` 先把配置、人体动作、物体几何、初始机器人状态都准备好，最后调用 `retargeter.retarget_motion(...)`。

## 2.1 数据导入：`load_motion_data(...)`

导入语句：

```python
human_joints, object_poses, smpl_scale = load_motion_data(
    task_type, data_format, data_path, task_name, constants, cfg.motion_data_config
)
```

把原始数据文件统一整理成后续 retargeting 使用的三个量：人体关节轨迹 `human_joints`、物体位姿轨迹 `object_poses`、以及人体到机器人尺度的缩放系数 `smpl_scale`。

#### 输入参数含义：

- `task_type`：任务类型，决定按哪条数据读取路径处理。当前脚本支持 `robot_only`、`object_interaction`、`climbing`。
- `data_format`：人体动作数据格式。已注册格式包括 `lafan`、`smplh`、`mocap`、`smplx`。
- `data_path`：输入数据目录。脚本会结合 `task_name` 在这个目录下寻找对应动作文件。
- `task_name`：动作序列名。
- `constants`：当前任务的常量设置。
- `cfg.motion_data_config`：当前人体数据格式的配置。

#### 返回值含义：

**`human_joints`**

`human_joints` 是人体关节在世界坐标系下的位置轨迹，形状为：

```python
human_joints.shape == (T, J, 3)
```

含义是：

```python
human_joints[t, j] = [x, y, z]
```

- `T`：动作帧数。
- `J`：人体关节数，取决于 `data_format`。例如 `smplh` 是 52 个关节，`lafan` 是 22 个关节，`mocap` 是 53 个关节，`smplx` 是 22 个关节。
- `3`：每个关节的三维位置。

**`object_poses`**

`object_poses` 是物体每一帧的刚体位姿，刚从 `load_motion_data(...)` 返回时形状为：

```python
object_poses.shape == (T, 7)
```

此时格式是：

```python
object_poses[t] = [qw, qx, qy, qz, x, y, z]
```

前 4 个数是物体姿态四元数；后 3 个数是物体位置。`robot_only` 和 `climbing` 会构造 dummy pose：

```python
object_poses[t] = [1, 0, 0, 0, 0, 0, 0]
```

注意：后续 `initialize_robot_pose(...)` 会把 `object_poses` 转成 MuJoCo 使用的顺序：

```python
[qw, qx, qy, qz, x, y, z] -> [x, y, z, qw, qx, qy, qz]
```

**`smpl_scale`**

`smpl_scale` 是一个浮点缩放系数，用来把人体动作尺度调整到机器人尺度附近。基本形式是：

```python
smpl_scale = robot_height / human_height
```

不同数据格式下来源略有不同：

- `smplh`：从 `demo_data/height_dict.pkl` 中按 subject 名查人体身高，再计算 `ROBOT_HEIGHT / human_height`。
- `lafan`：使用 `MotionDataConfig` 中的默认缩放系数：`1.27 / 1.7`。
- `mocap` / `climbing`：默认人体身高按 `1.78 m` 处理，计算 `ROBOT_HEIGHT / 1.78`。
- `smplx` 或兼容 `.npz` 格式：读取 `.npz["height"]`，计算 `ROBOT_HEIGHT / height`。

## 2.2 物体和场景几何准备：`setup_object_data(...)`

准备 interaction mesh 需要的物体点、地面点或地形点。输出是：

```python
object_local_pts, object_local_pts_demo, object_urdf_path
```

不同任务的含义不同：

- `robot_only`：没有交互物体，主要创建 ground points。
- `object_interaction`：加载物体 mesh，从表面采样 object points，并准备物体 URDF。
- `climbing`：使用 multi-box 地形相关的 URDF / XML / mesh 资源。

`object_local_pts_demo` 表示 demonstration 尺度下的物体点，`object_local_pts` 表示当前 retargeting 场景使用的物体点。
<!-- 启用 augmentation 时，两者可能不同。 -->

<!-- 注意这里处理的是物体本身的几何点或几何资源；`object_poses` 处理的是物体每一帧的位置和朝向。也就是说，物体“在哪里”由 `object_poses` 给出，物体“有多大、表面点在哪里”由 `setup_object_data(...)` 准备。 -->

注意：`object_interaction` 分支里，代码实际是 `object_local_pts = points`、`object_local_pts_demo = points * smpl_scale`。也就是说 target 侧用的是“缩放后的人 + 缩放后的箱子”，current 侧用的是“机器人 + 原始箱子”。如果理解为机器人要和同一个原始箱子交互，有尺度不一致问题。

## 2.3 Retargeter 创建：`build_retargeter_kwargs_from_config(...)` 和 `InteractionMeshRetargeter(...)`

`build_retargeter_kwargs_from_config(...)` 把 `cfg.retargeter` 中的选项整理成 `InteractionMeshRetargeter` 的初始化参数，例如：

- joint limits：默认开启。
- object non-penetration：默认开启。
- foot sticking：默认开启。
- foot lock window：默认不开启，需要设置 `retargeter.foot_lock.enable=True` 和锁定帧段。
- self-collision：默认不开启，需要设置 `retargeter.self_collision.enable=True` 和需要检查的 body pair。
- `visualize` / `debug`：默认不开启。
- `step_size`：局部优化每次允许更新的步长，默认 `0.2`。

然后脚本创建：

```python
retargeter = InteractionMeshRetargeter(**retargeter_kwargs)
```

这个对象内部会加载机器人模型、建立人体关节到机器人 link 的对应关系，并保存后续优化需要的约束和权重设置。

## 2.4 动作预处理：`preprocess_motion_data(...)`

把刚读入的人类示范数据变成当前机器人尺度下的输入：

- 根据脚趾最低点做高度归一化，让人体动作落到地面附近。
- 用 `smpl_scale` 缩放 `human_joints`。
- 如果有真实 `object_poses`，也会缩放物体位姿的平移部分：`x, y` 直接乘以 `scale`，`z` 不整体缩放，而是保持第一帧高度不变，只把相对第一帧的高度变化量乘以 `scale`。

<!-- - 对 `object_interaction` / `climbing`，额外返回 `object_moving_frame_idx`，表示物体第一次明显开始运动的帧。 -->

这里的“缩放物体位姿的平移部分”指的是只缩放物体位置 `[x, y, z]`，不缩放四元数 `[qw, qx, qy, qz]`。


## 2.5 机器人初值和物体位姿顺序：`initialize_robot_pose(...)`

根据第一帧人体姿态和物体位置初始化机器人根节点位姿，并处理 augmentation 情况。`augmentation` 默认不开启；只有显式设置 `augmentation=True` 时，才会生成扰动后的物体轨迹，并读取原始 retarget 结果作为参考轨迹。返回：

```python
q_init, q_nominal, object_poses_augmented, human_joints, object_poses
```

- `q_init`：优化开始时的机器人 qpos 初值。
- `q_nominal`：augmentation 模式下用于 nominal tracking 的参考轨迹；默认非 augmentation 时是 `None`。
- `object_poses_augmented`：augmentation 后的物体位姿；默认非 augmentation 时等于原始物体位姿。
- `human_joints`：预处理后的人体关节轨迹，通常已经完成地面对齐和 `smpl_scale` 缩放；这一步里一般原样返回，后面传给 `retarget_motion(...)` 生成 demo target。
- `object_poses`：会从 `[qw, qx, qy, qz, x, y, z]` 转成 MuJoCo 顺序 `[x, y, z, qw, qx, qy, qz]`。

<!-- 所以从这一步之后，传给 retargeter 的 `object_poses` 已经是 MuJoCo qpos 风格的顺序。 -->

## 2.6 脚接触序列：`extract_foot_sticking_sequence_velocity(...)`

根据人体脚趾关节的速度估计哪些帧脚应该保持贴地或相对固定。foot sticking 默认开启；如果关掉 `retargeter.activate_foot_sticking`，这段序列仍可计算，但优化里不会使用对应约束。

```python
foot_sticking_sequences = extract_foot_sticking_sequence_velocity(
    human_joints, retargeter.demo_joints, toe_names
)
```

## 2.7 核心优化入口：`retargeter.retarget_motion(...)`


```python
retargeter.retarget_motion(
    human_joint_motions=human_joints,
    object_poses=object_poses,
    object_poses_augmented=object_poses_augmented,
    object_points_local_demo=object_local_pts_demo,
    object_points_local=object_local_pts,
    foot_sticking_sequences=foot_sticking_sequences,
    q_a_init=q_init,
    q_nominal_list=q_nominal,
    original=not cfg.augmentation,
    dest_res_path=dest_res_path,
)
```


# 3. `interaction_mesh_retargeter.py` 的内部流程

`robot_retarget.py` 最后调用的 `retargeter.retarget_motion(...)`，实际实现在 `src/interaction_mesh_retargeter.py` 的 `InteractionMeshRetargeter` 类中。是 Holosoma 原始 interaction-mesh retargeting 的核心求解器：外层逐帧处理整段 motion，内层对每一帧做 SQP-style 的局部迭代，每次迭代用 CVXPY + Clarabel 求一个局部凸子问题。

流程：

```
InteractionMeshRetargeter.__init__(...)
  -> retarget_motion(...)
      for each frame:
        -> 构造 human/object interaction mesh target
        -> 计算 target Laplacian coordinates
        -> iterate(...)
            repeat local SQP iterations:
              -> solve_single_iteration(...)
                  -> 线性化 robot keypoint Laplacian
                  -> 加 foot / collision / joint limit / trust-region constraints
                  -> 求 CVXPY convex subproblem
                  -> 更新 q 并归一化 base quaternion
        -> 记录 qpos 和原始 nonlinear cost 评估项
  -> 保存 .npz 结果
```

## 3.1 初始化层：`__init__(...)`

`InteractionMeshRetargeter.__init__(...)` 做的事可以分成三类：加载场景、建立对应关系、准备优化参数。

第一类--加载场景。它根据任务类型选择 MuJoCo XML：

- `ground`：加载只有机器人和地面的 XML。
- `multi_boxes`：加载爬箱任务生成的 scene XML。
- 普通物体交互：加载机器人加物体的 XML，例如 `g1_29dof_w_largebox.xml`。

加载 XML 后会得到 `MjModel` 和 `MjData`。后面所有机器人 link 位置、Jacobian、碰撞距离，都再这个 MuJoCo 场景里算。

第二类--建立“人体点对应机器人点”的关系。

第三类--准备优化会用到的参数：

- `q_a_indices`：这一轮优化允许改 qpos 的哪些维度。
- `q_a_lb / q_a_ub`：这些变量的上下界。
- `laplacian_weights`、`smooth_weight`、`Q_diag`：目标函数权重。
- `step_size`：每次局部更新允许走多远。
- `foot_lock`、`self_collision` 等可选约束配置。

joint limits、object non-penetration、foot sticking 默认开启；foot lock、self-collision、visualize、debug 默认不开启。


## 3.2 外层 motion 循环：`retarget_motion(...)`

`retarget_motion(...)` 输入为：

```python
human_joint_motions          # 人体关节序列，形状 (T, J, 3)
object_poses                 # demo 里的物体位姿，MuJoCo 顺序
object_poses_augmented       # 当前机器人场景里的物体位姿，默认等于 object_poses
object_points_local_demo     # demo target 侧的物体表面点
object_points_local          # robot/current 侧的物体表面点
foot_sticking_sequences      # 每帧左右脚是否要防滑
q_a_init                     # 第一帧机器人初值
q_nominal_list               # augmentation 时的参考机器人轨迹，默认 None
```

先定义 `q_locked_list`：

- 如果没有 `q_nominal_list`，就创建空的 qpos 序列，并把第一帧可优化部分设成 `q_a_init`。
- 如果有 `q_nominal_list`，就用原始 retarget 结果作为参考初值。
- 然后把每一帧的最后 7 维设成 `object_poses_augmented`，也就是把物体位姿写进 qpos。

`q_locked_list[i]` 是第 `i` 帧完整 q 的参考背景：求解时会复制它，再用当前优化的 `q_a` 覆盖 `self.q_a_indices`。这样未优化分量和物体位姿有合理背景。
<!-- ，避免在全 0 或错误状态下计算 FK、Jacobian 和约束。 -->

之后按帧循环。第 `i` 帧主要做：

1. 从 `human_joint_motions[i]` 取出参与匹配的人体关节。
2. 如果有物体，把这些人体点转到物体局部坐标系。
3. 拼出 demo target 点集：

```python
source_vertices = np.vstack([human_mapped_joints_in_object, object_points_local_demo])
```

4. 用这些点建 interaction mesh，并计算这一帧的 `target_laplacian`。
5. 调用 `iterate(...)`，让机器人当前帧的 Laplacian 尽量接近这个 `target_laplacian`。
6. 保存这一帧求出的 qpos，并记录额外的 cost 评估量。

## 3.3 每帧内层迭代：`iterate(...)`

`iterate(...)` 帧内部的 SQP-style loop。重复调用 `solve_single_iteration(...)`：

```python
for _ in range(n_iter):
    q_a_n_last = q_n[self.q_a_indices]
    q_n, cost = self.solve_single_iteration(...)
    if np.isclose(cost, last_cost):
        break
```

第一帧默认最多迭代 50 次，后续帧默认最多迭代 10 次。每一次迭代都会在当前 `q_n` 附近重新线性化约束、目标函数，然后解一个局部凸子问题。

这里的 `q_t_last` 是上一帧最终解，用于 smoothness / foot sticking。

## 3.4 单次局部凸子问题：`solve_single_iteration(...)`

<!-- ### Laplacian 目标线性化

函数先用 `_calc_manipulator_jacobians(...)` 得到机器人匹配 link 的当前位置和 Jacobian。然后把机器人点和物体点拼成 vertices：

```python
vertices = np.vstack([robot_pts_local, obj_pts_local])
```

再根据 interaction mesh 的 adjacency list 计算 Laplacian matrix `L`：

```python
Kron = kron(L, I3)
J_L = Kron @ J_V
``` -->

<!-- 其中 `J_V` 是 robot vertices 对 `dqa` 的位置 Jacobian物体点在这一帧被锁定，所以它们对应的 Jacobian 为 0。 -->

<!-- 局部等式约束写成：

```python
J_L[:, self.q_a_indices] @ dqa - lap_var == -lap0_vec
``` -->

<!-- 目标：

```python
sum_squares(weight * (lap_var - target_lap_vec))
``` -->


### 约束项

`solve_single_iteration(...)` 会按开关配置加入这些约束：

- Foot sticking：如果人体脚在当前帧被判定为 sticking，则约束机器人对应脚 link 的 XY 位置保持在上一帧附近。
- Foot lock window：如果配置了显式锁脚窗口，则在指定 frame range 内约束脚 link 的 Z 接近 `z_floor`。默认不开启
- Object / ground non-penetration：调用 `_update_jacobians_and_phis_from_q(q)` 得到候选碰撞对的 signed distance `phi` 和距离方向 Jacobian `J`，加入线性不穿透约束。
- Self-collision：调用 `_compute_self_collision_constraints(frame_idx)`，对配置的机器人自碰撞 body pairs 加距离约束。
- Joint limits：约束 `dqa + q_a_n_last` 不超过 `q_a_lb / q_a_ub`。
- Trust region：用二阶锥约束限制步长：

```python
cp.SOC(self.step_size, dqa)
```

### 目标函数

局部目标由几部分组成：

- Laplacian match：匹配 interaction mesh 的 Laplacian coordinates。
- Nominal tracking：augmentation 模式下跟踪 `q_nominal_list` 的部分关节。
- Manual `Q_diag` regularization：对指定 q 维度施加手工权重。
- Smoothness：让当前帧增量接近上一帧到当前线性化点的差，减少时间抖动。

最后用：

```python
problem = cp.Problem(cp.Minimize(cp.sum(obj_terms)), constraints)
problem.solve(solver=cp.CLARABEL, ...)
```

如果第一帧带 SOC trust region 求解失败，代码会移除 SOC 约束再尝试一次。求解成功后：

```python
q_star[self.q_a_indices] = dqa_star + q_a_n_last
q_star[3:7] /= norm(q_star[3:7])
```

<!-- 也就是说，base quaternion 是先加性更新，再归一化。这个是当前实现行为，不是严格的流形指数映射更新。 -->

## 3.5 Jacobian 计算

`solve_single_iteration(...)` 里的约束需要很多一阶近似，例如“机器人某个点移动多少”“机器人和物体距离怎么变”。这些都要变成 Jacobian。

先区分两个变量：

```text
q    : MuJoCo qpos，位置变量，维度 nq。包含机器人根节点的位置，机器人根节点的朝向，机器人每个关节的角度。
v    : MuJoCo qvel，速度变量，维度 nv。包含机器人根节点的线速度，机器人根节点的角速度，机器人每个关节的角速度。
dq   : 优化器里的 qpos 增量，近似看成 qdot 的离散版本。
```

对普通关节，`qpos` 和 `qvel` 基本一一对应。对浮基：

```python
q_free = [x, y, z, qw, qx, qy, qz]      # 7 维 qpos
v_free = [vx, vy, vz, wx, wy, wz]       # 6 维 qvel
```


### qpos 增量到 qvel：`_build_transform_qdot_to_qvel_fast(...)`

这个函数构造矩阵：

$$
T(q) \in \mathbb{R}^{n_v \times n_q}
$$

使得：

$$
v = T(q)\dot q
$$

普通 hinge / slide joint 的块：

$$
v_i = \dot q_i
$$

所以对应矩阵元素就是 1。

free joint 的平移部分也是单位映射：

$$
\begin{bmatrix}v_x \\ v_y \\ v_z\end{bmatrix}
=
I_3
\begin{bmatrix}\dot x \\ \dot y \\ \dot z\end{bmatrix}
$$

<!-- free joint 的旋转部分需要把四元数导数转成角速度。 -->

设：

$$
Q = [q_w, q_x, q_y, q_z]^T
$$

代码使用：

$$
\omega = 2E(Q)\dot Q
$$

<!-- 其中 world-frame 版本的： -->

$$
E(Q)=
\begin{bmatrix}
-q_x & q_w & q_z & -q_y \\
-q_y & -q_z & q_w & q_x \\
-q_z & q_y & -q_x & q_w
\end{bmatrix}
$$

因此 free joint 的整体块是：

$$
\begin{bmatrix}
v_x \\ v_y \\ v_z \\ \omega_x \\ \omega_y \\ \omega_z
\end{bmatrix}
=
\begin{bmatrix}
I_3 & 0 \\
0 & 2E(Q)
\end{bmatrix}
\begin{bmatrix}
\dot x \\ \dot y \\ \dot z \\ \dot q_w \\ \dot q_x \\ \dot q_y \\ \dot q_z
\end{bmatrix}
$$

<!-- 如果场景里有动态物体，物体也有一个 free joint，代码会给物体的 free joint 再填一块同样的转换。 -->

### 点位置 Jacobian：`_calc_contact_jacobian_from_point(...)`

这个函数返回某个点的世界坐标对 `qpos` 增量的 Jacobian。

如果输入点是 body 局部坐标，先转到世界坐标：

$$
p_W = r_{WB} + R_{WB}p_B
$$

变量含义：

```text
p_B    : 点在 body 坐标系里的位置。
p_W    : 点在世界坐标系里的位置。
r_WB   : body 原点在世界系的位置。
R_WB   : body 坐标系到世界坐标系的旋转矩阵。
```

然后调用 MuJoCo：

```python
mujoco.mj_jac(model, data, Jp, Jr, p_W, body_idx)
```

MuJoCo 返回的 `Jp` 满足：

$$
\dot p_W = J_p^{(v)}(q)v
$$

其中：

$$
J_p^{(v)}(q) \in \mathbb{R}^{3 \times n_v}
$$

是“点速度对 MuJoCo qvel 的 Jacobian”。

<!-- 但优化器要的是“点位置变化对 qpos 增量的 Jacobian”。 -->

<!-- 因为： -->

猜测因为
$$
v = T(q)\dot q
$$

<!-- 所以： -->

$$
\dot p_W
= J_p^{(v)}(q)v
= J_p^{(v)}(q)T(q)\dot q
$$

函数最后返回：

$$
J_p^{(q)}(q)=J_p^{(v)}(q)T(q)
$$

<!-- 代码就是：

```python
T = self._build_transform_qdot_to_qvel_fast()
return Jp @ T
```

返回值形状是：

$$
J_p^{(q)} \in \mathbb{R}^{3 \times n_q}
$$

含义是：

$$
\Delta p_W \approx J_p^{(q)} \Delta q
$$ -->

### 碰撞距离和距离 Jacobian：`mj_geomDistance(...)` / `_compute_jacobian_for_contact_relative(...)`

涉及三个函数：

```text
_update_jacobians_and_phis_from_q(q)
  -> _prefilter_pairs_with_mj_collision(threshold)
  -> mujoco.mj_geomDistance(..., fromto)
  -> _compute_jacobian_for_contact_relative(..., fromto, dist)
```

#### `_prefilter_pairs_with_mj_collision(threshold)`

输入：

- `threshold`：碰撞检测距离阈值，来自 `self.collision_detection_threshold`。

输出：

- `candidates`：geom id pair 集合，形如 `{(g1, g2), ...}`。

代码临时把所有 geom 的 margin 设为 `threshold`，调用 `mujoco.mj_collision(m, d)`，再从 `d.contact` 里收集候选 pair：

```python
m.geom_margin[:] = threshold
mujoco.mj_collision(m, d)
candidates.add((min(g1, g2), max(g1, g2)))
m.geom_margin[:] = self._saved_margins
```


#### `_update_jacobians_and_phis_from_q(q)`

输入：

- `q`：当前 SQP / inner iteration 的 full MuJoCo `qpos` 线性化点。

输出：

- `Js`：字典，`Js[(g1, g2)] = J_{\phi,c}(q)`，是该 collision pair 的 signed-distance Jacobian，定义在 full `qpos` 上。
- `phis`：字典，`phis[(g1, g2)] = \phi_c(q)`，是该 collision pair 的 signed distance。


调用 `_prefilter_pairs_with_mj_collision(threshold)` 得到候选集合。再调用精确距离查询：

```python
fromto[:] = 0.0
dist = mujoco.mj_geomDistance(m, d, g1, g2, threshold, fromto)
```

`dist` 是 MuJoCo 返回的 signed distance。约定是：`dist > 0` 表示分离，`dist = 0` 表示刚好接触，`dist < 0` 表示穿透；`fromto` 会被原地重新写入，新写入的内容定义了：

$$
p_1(q),\quad p_2(q)
$$

对每个保留下来的 pair $c=(g_1,g_2)$，这个函数调用`_compute_jacobian_for_contact_relative`生成：

$$
\phi_c(q)=\operatorname{sdist}(g_1(q),g_2(q))
$$

$$
J_{\phi,c}(q)=\frac{\partial \phi_c}{\partial q}(q)
$$

并存入：

```python
Js[(g1, g2)] = J_rel
phis[(g1, g2)] = float(dist)
```

#### `_compute_jacobian_for_contact_relative(geom1, geom2, geom1_name, geom2_name, fromto, dist)`

输入：

- `geom1`, `geom2`：MuJoCo geom 对象，用来拿 `bodyid`。
- `geom1_name`, `geom2_name`：geom 名字，用于代码中的判断语句。
- `fromto`：`mj_geomDistance(...)` 原地写入的 6 维数组，`fromto[:3] = p_1`，`fromto[3:] = p_2`。
- `dist`：`mj_geomDistance(...)` 返回的 signed distance，即 $\phi$。

输出：

- `J_phi`：一维数组，形状是 `(nq,)`，表示 signed distance 对 full `qpos` 的 Jacobian。

如果 $\|p_1-p_2\| > 1e-12$，用链式法则有：

$$
\hat n = \operatorname{sign}(\phi)\frac{p_1-p_2}{\|p_1-p_2\|},\qquad
J_\phi = \hat n^T(J_1-J_2)
$$


如果 $\|p_1-p_2\|$ 太小，连线方向本身不稳定，：如果有一侧是 `ground`，就用竖直方向做法向；否则把这次 pair 的法向设成 0，也就是让这次 Jacobian 退化成 0，避免数值爆炸。两个最近点的 `qpos` Jacobian 分别由 `_calc_contact_jacobian_from_point(..., input_world=True)` 给出：

$$
J_1 = \frac{\partial p_1}{\partial q},\quad
J_2 = \frac{\partial p_2}{\partial q}
$$



代码对应：

```python
J_bodyA = _calc_contact_jacobian_from_point(...)
J_bodyB = _calc_contact_jacobian_from_point(...)
Jc = J_bodyA - J_bodyB
J_phi = nhat_BA_W @ Jc
```

`J_phi` 是 full `qpos` 上的一维 Jacobian。进入 convex 子问题时，代码只截取 active variables：

```python
Ja_n = Ja_n_full[self.q_a_indices]
```

最后使用 signed distance 的一阶线性化：

$$
\phi(q + \Delta q) \approx \phi(q) + J_\phi \Delta q
$$

对 object / ground non-penetration，代码允许 `penetration_tolerance` 范围内的轻微穿透：

$$
\phi(q) + J_{\phi}\Delta q \ge -\epsilon_{pen}
$$

整理成 CVXPY 约束是：

$$
J_{\phi}\Delta q \ge -\phi(q)-\epsilon_{pen}
$$

对应实现：

```python
rhs = -phi - self.penetration_tolerance
constraints += [Ja_n @ dqa >= rhs]
```

对应代码里的右端项是：

```python
rhs = self._self_collision_tolerance - phi
constraints += [Ja_n @ dqa >= rhs]
```

<!-- ## 3.6 输出和调试信息

`retarget_motion(...)` 最终保存的不只是 qpos：

- `qpos`：retargeted robot/object qpos trajectory。
- `human_joints`：输入的人体关节轨迹。
- `fps`：固定写为 30。
- `cost`：最后一次局部求解 cost。
- `original_lap_cost`、`original_smooth_cost`、`original_lap_smooth_cost`：在最终帧解上额外评估的 raw nonlinear laplacian / smooth cost，用于跨 solver variant 的比较。 -->


### 其它
如果 `visualize=True` 或 `debug=True`，类里还有大量 Viser 绘制函数，例如 `draw_q(...)`、`draw_keypoints(...)`、`visualize_motion(...)`。


## 3.4 四元数归一化扩张下的欧氏雅可比

设

$$
Q\in \mathbb{R}^4\setminus\{0\},
\qquad
N(Q)=\frac{Q}{\|Q\|_2}.
$$

物理姿态变量不是整个 \(\mathbb{R}^4\)，而是单位球面

$$
S^3=\{Q\in\mathbb{R}^4:\|Q\|_2=1\}.
$$

令点位置映射为

$$
p:S^3\times\mathbb{R}^m\to\mathbb{R}^3,
\qquad
(Q,y)\mapsto p(Q,y).
$$

若把四元数坐标作为四维欧氏变量讨论，则需要先指定 \(p\) 在非单位四元数处的扩张。取归一化扩张

$$
\hat p:\left(\mathbb{R}^4\setminus\{0\}\right)\times\mathbb{R}^m\to\mathbb{R}^3,
\qquad
\hat p(Q,y)=p(N(Q),y)
=p\left(\frac{Q}{\|Q\|_2},y\right).
$$

归一化映射的导数为

$$
DN(Q)=\frac{1}{\|Q\|_2}
\left(I_4-\frac{QQ^\top}{\|Q\|_2^2}\right).
$$

在 \(Q\in S^3\) 处，\(\|Q\|_2=1\)，于是

$$
DN(Q)=I_4-QQ^\top.
$$

该矩阵是到切空间

$$
T_QS^3=\{\eta\in\mathbb{R}^4:Q^\top\eta=0\}
$$

的正交投影。任意 \(\delta Q\in\mathbb{R}^4\) 有分解

$$
\delta Q=(I_4-QQ^\top)\delta Q+(Q^\top\delta Q)Q,
$$

其中第一项属于 \(T_QS^3\)，第二项属于径向子空间 \(\operatorname{span}\{Q\}\)。因此

$$
D\hat p(Q,y)[\delta Q,\delta y]
=
Dp(Q,y)[(I_4-QQ^\top)\delta Q,\delta y],
$$

并且

$$
D\hat p(Q,y)[Q,0]=0.
$$

记 \(J_p(q)\) 为点位置对切速度坐标的雅可比。记 \(T(q)\) 为从欧氏坐标扰动到切速度坐标的线性映射。对四元数块，设

$$
T_Q(Q)\delta Q=2E(Q)\delta Q.
$$

其中 \(E(Q)Q=0\)，故

$$
T_Q(Q)Q=0,
\qquad
T_Q(Q)\delta Q=T_Q(Q)(I_4-QQ^\top)\delta Q.
$$

在切空间方向上，速度雅可比的定义给出

$$
Dp(q)[\delta q]=J_p(q)T(q)\delta q,
\qquad
\delta q\in T_q\mathcal{Q}.
$$

现在令 \(\delta q\in\mathbb{R}^{n_q}\) 任意。四元数扰动的切向部分由 \(D N(Q)\) 保留，径向部分同时被 \(D\hat p(q)\) 与 \(J_p(q)T(q)\) 消去。于是

$$
D\hat p(q)\delta q=J_p(q)T(q)\delta q,
\qquad
\forall\delta q\in\mathbb{R}^{n_q}.
$$

因此得到完整欧氏矩阵恒等式

$$
D\hat p(q)=J_p(q)T(q).
$$

这个等式中的 \(D\hat p(q)\) 是归一化扩张的雅可比。若取另一个不经归一化的扩张

$$
p_{\mathrm{raw}}:\mathbb{R}^{n_q}\to\mathbb{R}^3,
$$

则由

$$
\left(Dp_{\mathrm{raw}}(q)-J_p(q)T(q)\right)\delta q=0,
\qquad
\delta q\in T_q\mathcal{Q},
$$

只能推出二者在切空间上的限制相同，不能推出

$$
Dp_{\mathrm{raw}}(q)=J_p(q)T(q)
$$

作为 \(\mathbb{R}^{n_q}\to\mathbb{R}^3\) 的矩阵恒等式。
